# Australia: Aldi shelf prices and this week's Special Buys

Two runs of
[Aldi Australia Price Scraper: Shelf Prices & Special Buys](https://apify.com/yappman/au-supermarket-price-scraper):

1. A keyword search at Aldi Australia, sorted by unit price.
2. Aldi's Special Buys drops dated this week (Monday to Sunday, Sydney time).

Coles and Woolworths are not available at the moment. Their rows are read only on runs started by the Actor owner's
account (from version 0.1.15), so this example runs Aldi only.

**Cost at Apify Free-tier prices:** at most **$0.115** in total.

| Run | Rows (`maxItems`) | Rows | Start | Most it can cost |
|---|---|---|---|---|
| Search | 10 | $0.015 | $0.005 | $0.02 |
| Special Buys | 60 | $0.09 | $0.005 | $0.095 |

Each row is $0.0015 on the Free tier.

More data: [data.yappman.com](https://data.yappman.com).

In [ ]:
import os
import time
from datetime import timedelta
from decimal import Decimal
from pathlib import Path

import pandas as pd
from apify_client import ApifyClient

TOKEN = os.environ.get("APIFY_TOKEN")
if not TOKEN:
    raise RuntimeError("Set the APIFY_TOKEN environment variable first. See the README.")
client = ApifyClient(TOKEN)
pd.set_option("display.html.use_mathjax", False)  # show third-party text as text

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)  # git-ignored: nothing you collect is committed


def run_actor(actor_id: str, run_input: dict, max_charge_usd: float):
    """Start an Actor, wait for it, and return the finished run.

    Spend is capped twice: maxItems in run_input limits the rows saved, and
    max_total_charge_usd stops the run if its charges would pass the cap.
    The run times out after 10 minutes.
    """
    run = client.actor(actor_id).call(
        run_input=run_input,
        max_total_charge_usd=Decimal(str(max_charge_usd)),
        run_timeout=timedelta(minutes=10),
    )
    if run is None or run.status != "SUCCEEDED":
        raise RuntimeError(f"{actor_id} did not succeed: {None if run is None else run.status}")
    print(f"{actor_id}: run {run.id}")
    # Charges settle a few seconds after the run ends, so read the record again before printing them.
    time.sleep(5)
    settled = client.run(run.id).get()
    print("Charged events:", settled.charged_event_counts)
    return run

## 1. Search Aldi Australia

`retailers` is `["aldi_au"]`, the Actor's default. The search covers one keyword
and saves at most 10 rows. Rows with no `price` are dropped, and the rest are
sorted by unit price.

In [ ]:
ACTOR = "yappman/au-supermarket-price-scraper"

# Free tier: 10 rows x $0.0015 + 1 start x $0.005 = $0.02
search_run = run_actor(
    ACTOR,
    {"mode": "search", "retailers": ["aldi_au"], "queries": ["full cream milk"], "maxItems": 10},
    max_charge_usd=0.025,
)
df = pd.DataFrame(client.dataset(search_run.default_dataset_id).iterate_items())
df = df[df["price"].notna()].copy()
df["unitValue"] = df["unitPrice"].apply(lambda u: (u or {}).get("value"))
df["unitMeasure"] = df["unitPrice"].apply(lambda u: (u or {}).get("measure"))
df["reduced"] = df["wasPrice"].notna() & (df["wasPrice"] > df["price"])
print(len(df), "products")

In [ ]:
df.to_csv(OUTPUT / "au_aldi_search.csv", index=False)
df.sort_values("unitValue")[["name", "packSize", "price", "wasPrice", "unitValue", "unitMeasure", "reduced"]]

## 2. This week's Aldi Special Buys

The run asks for every date in the current Sydney week. Aldi drops on set days,
and dates Aldi's page does not list are skipped and named in the run's
`RUN_SUMMARY` record.

In [ ]:
today = pd.Timestamp.now(tz="Australia/Sydney").normalize()
monday = today - pd.Timedelta(days=today.weekday())
week = [(monday + pd.Timedelta(days=i)).strftime("%Y-%m-%d") for i in range(7)]

# Free tier: 60 rows x $0.0015 + 1 start x $0.005 = $0.095
sb_run = run_actor(
    ACTOR,
    {"mode": "specialBuys", "specialBuysDrops": week, "maxItems": 60},
    max_charge_usd=0.10,
)
sb = pd.DataFrame(client.dataset(sb_run.default_dataset_id).iterate_items())

summary = client.key_value_store(sb_run.default_key_value_store_id).get_record("RUN_SUMMARY")
if summary:
    print("Special Buys summary:",
          summary["value"].get("perRetailer", {}).get("aldi_au", {}).get("specialBuys"))

if sb.empty:
    print("No Special Buys listed for", week[0], "to", week[-1])
else:
    sb["category"] = sb["categoryPath"].apply(lambda p: p[0] if p else None)
    print(sb.groupby("onSaleDate").size().rename("products"))

In [ ]:
if not sb.empty:
    sb.to_csv(OUTPUT / "au_special_buys_this_week.csv", index=False)
    display(sb.sort_values(["onSaleDate", "price"])
              [["onSaleDate", "name", "price", "category", "promotionText"]])

## Notes

* Aldi publishes no barcode, so Aldi rows carry an empty `ean`. This example does not
  match products across shops.
* Coles and Woolworths are not available at the moment. They are read only on runs started by the Actor owner's account.
  On other accounts those shops are skipped.
* Rows belong to the Actor's data terms, not to this repository's MIT licence.
  The CSVs are written to `output/`, which git ignores.